# SageMaker Studio Demo: Bike Sharing Demand

- Dataset: [Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) (UCI)
- Regression: the target `cnt` is a count of rides per hour

Steps

1. setup notebook, install Python libraries
2. load `raw/hour.csv` from S3
3. engineer features, write to `featured/`
4. train model
   - baseline
   - deeper tree
5. evaluate, compare, and visualize
6. write the artifact to `model/`


---

## 1. Setup

The Studio image ships pandas, scikit-learn, boto3 and joblib. Only
parquet and plotting need installing.


In [ ]:
%pip install --quiet pyarrow matplotlib

Get the bucket name from terraform:

```
terraform -chdir=infra output -raw data_bucket
```


In [ ]:
import io

import boto3
import numpy as np
import pandas as pd

REGION = "ca-central-1"
BUCKET = "mlops-sagemaker-studio-dev-data-3vi8kw"

s3 = boto3.client("s3", region_name=REGION)
s3.head_bucket(Bucket=BUCKET)

# Only alice may write the shared featured/ and model/ prefixes; bob is
# explicitly denied and works under users/bob/. The execution role is the
# one identity always present in a space, so derive the prefix from it
# and keep one notebook runnable as either user.
caller = boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"]
USER = "alice" if "role-alice" in caller else "bob"
WRITE_PREFIX = "" if USER == "alice" else f"users/{USER}/"

RAW_KEY = "raw/hour.csv"
FEATURED_KEY = f"{WRITE_PREFIX}featured/hour.parquet"
MODEL_KEY = f"{WRITE_PREFIX}model/model.joblib"
FEATURES_KEY = f"{WRITE_PREFIX}model/features.joblib"

print(f"s3://{BUCKET} reachable")
print(f"caller: {caller}")
print(f"{USER} -> writing to s3://{BUCKET}/{WRITE_PREFIX or '<shared>'}")

---

## 2. Load `raw/hour.csv`

17,379 hourly records from 2011-2012.


In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=RAW_KEY)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print(f"{len(df)} rows, {df.dteday.min()} -> {df.dteday.max()}")
df.head()

---

## 3. Engineer features

`casual + registered == cnt` in every row, so both leak the target.
`instant` is a row index, `dteday` is already covered by the calendar
columns. Drop all four.


In [ ]:
TARGET = "cnt"
DROP = ["instant", "dteday", "casual", "registered", TARGET]
FEATURES = [c for c in df.columns if c not in DROP]

leak = (df.casual + df.registered != df[TARGET]).sum()
print(f"rows where casual + registered != cnt: {leak}")
print(f"{len(FEATURES)} features: {FEATURES}")

Write to `featured/`. Alice writes the shared prefix; bob writes his own
copy under `users/bob/`.

In [ ]:
featured = df[FEATURES + [TARGET]]

buf = io.BytesIO()
featured.to_parquet(buf, index=False)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, FEATURED_KEY)

print(f"{featured.shape[0]} rows x {featured.shape[1]} cols")
print(f"s3://{BUCKET}/{FEATURED_KEY}")

### Split by time

Train on 2011, test on 2012. A random split would let the model see
hours next to the ones it is scored on.


In [ ]:
train = featured[featured.yr == 0]
test = featured[featured.yr == 1]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")

---

## 4. Train

Two random forests on the same split:

- **baseline** -- `min_samples_leaf=5`
- **deeper** -- `min_samples_leaf=1`, fully grown trees


In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


RUNS = {
    "baseline": {"n_estimators": 100, "min_samples_leaf": 5},
    "deeper": {"n_estimators": 100, "min_samples_leaf": 1},
}

models = {}
results = {}

for name, params in RUNS.items():
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(X_train, y_train)

    metrics = evaluate(y_test, model.predict(X_test))

    buf = io.BytesIO()
    joblib.dump(model, buf)
    metrics["size_mb"] = buf.tell() / 1024 / 1024

    models[name], results[name] = model, metrics
    print(f"{name:9} rmse={metrics['rmse']:6.1f}  mae={metrics['mae']:5.1f}  "
          f"r2={metrics['r2']:.3f}  {metrics['size_mb']:.1f}MB")

---

## 5. Evaluate, compare, and visualize

An rmse of 126 means nothing on its own. Predicting the training mean
every hour is the floor to beat.


In [ ]:
floor = evaluate(y_test, np.full(len(y_test), y_train.mean()))

table = pd.DataFrame(results).T
table.loc["predict-the-mean"] = {**floor, "size_mb": 0.0}
table["vs_floor_%"] = (floor["rmse"] - table["rmse"]) / floor["rmse"] * 100

table.round(2)

The deeper tree is 0.9% better on rmse and 6x the size. That is the
trade-off, and it is why the baseline is the one saved above.


### Visualize - Baseline


In [ ]:
import matplotlib.pyplot as plt

BEST = "baseline"
model = models[BEST]

pred = model.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle(f"Bike sharing demand -- 2012 holdout ({BEST})")

# what drives demand
ax = axes[0]
imp = pd.Series(model.feature_importances_, index=FEATURES).nlargest(8)
ax.barh(imp.index[::-1], imp.values[::-1])
ax.set_title("feature importance")

# predicted vs actual
ax = axes[1]
ax.scatter(y_test, pred, s=4, alpha=0.2)
lim = [0, max(y_test.max(), pred.max())]
ax.plot(lim, lim, ls="--", c="grey", lw=1)
ax.set_xlabel("actual")
ax.set_ylabel("predicted")
ax.set_title(f"predicted vs actual (r2={results[BEST]['r2']:.3f})")

# average day
ax = axes[2]
by_hour = pd.DataFrame({"hr": X_test.hr, "actual": y_test, "predicted": pred})
by_hour = by_hour.groupby("hr").mean()
ax.plot(by_hour.index, by_hour.actual, label="actual", lw=2)
ax.plot(by_hour.index, by_hour.predicted, label="predicted", lw=2, ls="--")
ax.set_xlabel("hour of day")
ax.set_ylabel("mean rides")
ax.set_title("average day")
ax.set_xticks(range(0, 24, 3))
ax.legend()

fig.tight_layout()
plt.show()

`hr` dominates. The model follows the 8am and 5-6pm commute peaks but
undershoots the tallest ones -- 2012 ridership grew past anything in the
2011 training data.


---

### 6. Write the artifact to `model/`

The baseline is the one worth keeping -- 0.9% worse on rmse, 6x smaller.
The feature list goes with it, because inference has to rebuild the
input columns in this exact order.

Alice writes the shared `model/`, which is what deployment serves. Bob
writes `users/bob/model/` -- he proposes a model, alice's stays
authoritative.

In [ ]:
def upload(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    buf.seek(0)
    s3.upload_fileobj(buf, BUCKET, key)
    print(f"s3://{BUCKET}/{key}")


upload(model, MODEL_KEY)
upload(FEATURES, FEATURES_KEY)